# Native quantum programming, part II

Work through each exercise and compare with its executable FlagQuantum solution.

## Distinguish coherent programs from host-side composition

The first pass runs each supplied solution with independent numerical checks.
The solution cells then expose editable qfunc definitions and synthesize them
directly. Product phases and the linear predicate are coherent operations;
`within`/`apply` cleans the local computed value after use. Fourier addition is
modular at its declared width. QPE returns a fractional register with a declared
bit resolution; it is not an arbitrary-precision classical estimate.

Two later exercises intentionally use Circuit: the host callback iterates over
physical wires, and the symbolic ansatz binds a reusable runtime template. Those
are distinct from static qfunc callbacks and classical specialization. No
parameterized qfunc runtime reuse is claimed. Try a nonzero change to one phase
angle and predict the interference before editing both the program and its
reference. Compare complex amplitudes, not only their probabilities.


Set up the local FlagQuantum session.

In [ ]:
from ast import FunctionDef, parse
from inspect import getsource

import flagquantum as fq
from examples.synthesis.tutorials.basics.quantum_programming.qmod_tutorial_part2.overview import (
    quantum,
    reference,
    workflow,
)


def definition(name: str) -> str:
    source = getsource(quantum).splitlines()
    for node in parse(getsource(quantum)).body:
        if (
            isinstance(node, FunctionDef)
            and node.name == name
            and node.end_lineno is not None
        ):
            start = min(
                (item.lineno for item in node.decorator_list), default=node.lineno
            )
            return chr(10).join(source[start - 1 : node.end_lineno])
    raise ValueError(name)


import math
from typing import Annotated
from flagquantum import synth
from flagquantum.synth import (
    H,
    Output,
    QArray,
    QBit,
    QNum,
    X,
    allocate,
    apply,
    control,
    hadamard_transform,
    phase,
    qft,
    qfunc,
    qpe,
    suzuki_trotter,
    within,
)

HAMILTONIAN = quantum.HAMILTONIAN


7A. Encode the product of two two-bit numbers as a phase.

In [ ]:
product, result, checked = workflow.run_native(
    "product_phase", reference.phase_state(), ("x", "y"), max_qubits=4, angles=True
)
print(checked.vector.tolist())
print(product.report["resources"])


7B. Mark the two solutions of 3a + b = 9 without changing their probabilities.

In [ ]:
oracle, result, marked = workflow.run_native(
    "solution_oracle", reference.phase_state(oracle=True), ("a", "b"), max_qubits=4
)
print(marked.vector.tolist())


8. Compute 2x + 1 in a local register. Flip the output if the value exceeds three, then uncompute.

In [ ]:
comparison, result, local = workflow.run_native(
    "local_comparison",
    reference.comparison_state(),
    ("x", "res"),
    max_qubits=7,
    max_workspace=4,
)
print(workflow.read_counts(comparison, result, ("x", "res"), seed=10))
print("Scratch leakage:", local.workspace_probability)


9A. Add a three-bit quantum number to a four-bit accumulator initialized to two.

In [ ]:
addition, result, simple_add = workflow.run_native(
    "fourier_add",
    reference.addition_state(controlled=False),
    ("x", "y"),
    max_qubits=7,
    angles=True,
)
print(workflow.read_counts(addition, result, ("x", "y"), seed=11))


9B. Control only the phase block. The Fourier transform and its inverse surround both branches.

In [ ]:
controlled, result, guarded_add = workflow.run_native(
    "controlled_fourier_add",
    reference.addition_state(controlled=True),
    ("ctrl", "x", "y"),
    max_qubits=8,
)
print(workflow.read_counts(controlled, result, ("ctrl", "x", "y"), seed=12))


10A. Apply a two-step, second-order Suzuki–Trotter formula at time three on four qubits.

In [ ]:
evolution, result, evolved = workflow.run_native(
    "pauli_evolution", reference.trotter_state(), ("qarr",), max_qubits=4
)
print(evolution.report["resources"])
print(workflow.read_counts(evolution, result, ("qarr",), seed=13))


10B. Observe X(0) + X(1) on the prepared plus pair.

In [ ]:
plus, plus_error = workflow.observe_plus()
print(plus, plus_error)


11A. Pass a one-qubit operation to a reusable Python function and apply it to three qubits.

In [ ]:
callback_counts, callback_error = workflow.read_callback()
print(callback_counts, callback_error)


11B. Use a controlled RZ phase callback and four fractional phase bits.

In [ ]:
phase_probabilities, phase_state = workflow.read_qpe()
print(dict(phase_probabilities), phase_state.error)


12A. Build one symbolic template for both sampling bindings.

In [ ]:
template = quantum.parameterized_ansatz()
print(fq.__name__, template.parameter_names)


12B. Define a third binding for the observable experiment on the same template.

In [ ]:
print(getsource(workflow.observe_binding))


Solution 7A. Decompose the bilinear phase by bit significance using four controlled phases.

In [ ]:
@qfunc
def product_phase(
    angles: list[float], x: Output[Annotated[QNum, 2]], y: Output[Annotated[QNum, 2]]
) -> None:
    allocate(x)
    allocate(y)
    hadamard_transform(x)
    hadamard_transform(y)
    # The compiler expands each callback in its current static loop binding.
    for i in range(2):
        for j in range(2):
            control(x[i], lambda: phase(y[j], angles[i + j]))  # noqa: B023


editable_program = synth.synthesize(
    product_phase,
    classical_inputs={"angles": [math.pi * (1 << i) / 8 for i in range(6)]},
    max_qubits=4,
    max_workspace=0,
    max_operations=180,
    memory_limit_bytes=1024**2,
)
editable_result = fq.run(editable_program)
editable_wires = tuple(
    wire for name in ("x", "y") for wire in editable_program.registers[name]
)
editable_check = workflow.compare(
    editable_result.statevector(), reference.phase_state(), editable_wires
)
print("Edited solution amplitude error:", editable_check.error)
print("Edited solution workspace leakage:", editable_check.workspace_probability)

print("Product phase maximum amplitude error:", checked.error)


Solution 7B. Use the predicate to apply a phase of pi.

In [ ]:
@qfunc
def mark(a: QNum, b: QNum) -> None:
    control(3 * a + b == 9, lambda: phase(1, math.pi))


@qfunc
def solution_oracle(
    a: Output[Annotated[QNum, 2]], b: Output[Annotated[QNum, 2]]
) -> None:
    allocate(a)
    allocate(b)
    hadamard_transform(a)
    hadamard_transform(b)
    mark(a, b)


editable_program = synth.synthesize(
    solution_oracle,
    classical_inputs=None,
    max_qubits=4,
    max_workspace=0,
    max_operations=180,
    memory_limit_bytes=1024**2,
)
editable_result = fq.run(editable_program)
editable_wires = tuple(
    wire for name in ("a", "b") for wire in editable_program.registers[name]
)
editable_check = workflow.compare(
    editable_result.statevector(), reference.phase_state(oracle=True), editable_wires
)
print("Edited solution amplitude error:", editable_check.error)
print("Edited solution workspace leakage:", editable_check.workspace_probability)

print("Predicate maximum amplitude error:", marked.error)


Solution 8. Use within/apply to give the local register a verified inverse.

In [ ]:
@qfunc
def local_comparison(x: Output[Annotated[QNum, 2]], res: Output[QBit]) -> None:
    allocate(x)
    allocate(res)
    hadamard_transform(x)
    with within():
        tmp: Annotated[QNum, 3] = 2 * x + 1
    with apply():
        control(tmp > 3, lambda: X(res))


editable_program = synth.synthesize(
    local_comparison,
    classical_inputs=None,
    max_qubits=7,
    max_workspace=4,
    max_operations=180,
    memory_limit_bytes=1024**2,
)
editable_result = fq.run(editable_program)
editable_wires = tuple(
    wire for name in ("x", "res") for wire in editable_program.registers[name]
)
editable_check = workflow.compare(
    editable_result.statevector(), reference.comparison_state(), editable_wires
)
print("Edited solution amplitude error:", editable_check.error)
print("Edited solution workspace leakage:", editable_check.workspace_probability)

assert local.workspace_probability == 0


Solution 9A. The inverse transform translates phase into modular addition.

In [ ]:
@qfunc
def fourier_add(
    angles: list[float], x: Output[Annotated[QNum, 3]], y: Output[Annotated[QNum, 4]]
) -> None:
    allocate(x)
    allocate(y)
    hadamard_transform(x)
    X(y[1])
    with within():
        qft(y)
    with apply():
        for i in range(3):
            for j in range(4):
                control(x[i], lambda: phase(y[j], angles[i + j]))  # noqa: B023


editable_program = synth.synthesize(
    fourier_add,
    classical_inputs={"angles": [math.pi * (1 << i) / 8 for i in range(6)]},
    max_qubits=7,
    max_workspace=0,
    max_operations=180,
    memory_limit_bytes=1024**2,
)
editable_result = fq.run(editable_program)
editable_wires = tuple(
    wire for name in ("x", "y") for wire in editable_program.registers[name]
)
editable_check = workflow.compare(
    editable_result.statevector(),
    reference.addition_state(controlled=False),
    editable_wires,
)
print("Edited solution amplitude error:", editable_check.error)
print("Edited solution workspace leakage:", editable_check.workspace_probability)

print("Addition maximum amplitude error:", simple_add.error)


Solution 9B. Superpose the control and check both branches including their relative phase.

In [ ]:
@qfunc
def controlled_fourier_add(
    ctrl: Output[QBit], x: Output[Annotated[QNum, 3]], y: Output[Annotated[QNum, 4]]
) -> None:
    allocate(ctrl)
    allocate(x)
    allocate(y)
    H(ctrl)
    hadamard_transform(x)
    X(y[1])
    with within():
        qft(y)
    with apply():
        control(ctrl, lambda: phase(x * y, math.pi / 8))


editable_program = synth.synthesize(
    controlled_fourier_add,
    classical_inputs=None,
    max_qubits=8,
    max_workspace=0,
    max_operations=180,
    memory_limit_bytes=1024**2,
)
editable_result = fq.run(editable_program)
editable_wires = tuple(
    wire for name in ("ctrl", "x", "y") for wire in editable_program.registers[name]
)
editable_check = workflow.compare(
    editable_result.statevector(),
    reference.addition_state(controlled=True),
    editable_wires,
)
print("Edited solution amplitude error:", editable_check.error)
print("Edited solution workspace leakage:", editable_check.workspace_probability)

print("Controlled addition maximum amplitude error:", guarded_add.error)


Solution 10A. The independent reference uses dense Pauli exponentials for the same product formula.

In [ ]:
@qfunc
def pauli_evolution(qarr: Output[QArray[QBit]]) -> None:
    allocate(4, qarr)
    suzuki_trotter(HAMILTONIAN, 3.0, 2, 2, qarr)


editable_program = synth.synthesize(
    pauli_evolution,
    classical_inputs=None,
    max_qubits=4,
    max_workspace=0,
    max_operations=180,
    memory_limit_bytes=1024**2,
)
editable_result = fq.run(editable_program)
editable_wires = tuple(
    wire for name in ("qarr",) for wire in editable_program.registers[name]
)
editable_check = workflow.compare(
    editable_result.statevector(), reference.trotter_state(), editable_wires
)
print("Edited solution amplitude error:", editable_check.error)
print("Edited solution workspace leakage:", editable_check.workspace_probability)

print("Trotter maximum amplitude error:", evolved.error)


Solution 10B. Request the observable expectation on the native prepared circuit.

In [ ]:
print(getsource(workflow.observe_plus))
print("X(0) + X(1):", plus)


Solution 11A. Use a typed Python callable at circuit construction.

In [ ]:
print(getsource(quantum.apply_to_all))
print("Finite three-qubit samples:", callback_counts)


Solution 11B. A conditional phase polynomial exactly expresses CRZ(pi), including the other eigenphase.

In [ ]:
@qfunc
def estimate_phase(
    theta: Output[Annotated[QNum, (4, False, 4)]], state: Output[QArray[QBit]]
) -> None:
    allocate(theta)
    allocate(2, state)
    X(state[0])
    X(state[1])
    qpe(lambda: phase(state[0] * (2 * state[1] - 1), math.pi / 2), theta)


editable_program = synth.synthesize(
    estimate_phase,
    classical_inputs=None,
    max_qubits=6,
    max_workspace=0,
    max_operations=180,
    memory_limit_bytes=1024**2,
)
editable_result = fq.run(editable_program)
editable_wires = tuple(
    wire for name in ("theta", "state") for wire in editable_program.registers[name]
)
editable_check = workflow.compare(
    editable_result.statevector(), reference.qpe_state(), editable_wires
)
print("Edited solution amplitude error:", editable_check.error)
print("Edited solution workspace leakage:", editable_check.workspace_probability)

print("Decoded fractional phase:", dict(phase_probabilities))


Solution 12A. Bind and sample both settings without rebuilding the symbolic template.

In [ ]:
samples = workflow.read_bindings(template)
print("Finite samples from the two bindings:", samples)
print("Unchanged symbolic template:", template.parameter_names)


Solution 12B. Bind the same template for a third observable measurement.

In [ ]:
energy, energy_error = workflow.observe_binding(template)
print("Bound expectation:", energy, "maximum amplitude error:", energy_error)
